In [25]:
import pandas as pd
import numpy as np

In [26]:
df=pd.read_csv('powerplant_data.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9568 entries, 0 to 9567
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AT      9568 non-null   float64
 1   V       9568 non-null   float64
 2   AP      9568 non-null   float64
 3   RH      9568 non-null   float64
 4   PE      9568 non-null   float64
dtypes: float64(5)
memory usage: 373.9 KB


In [27]:
df.head(5)

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


In [28]:
df.isnull().sum()

AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64

In [29]:
x=df.drop('PE',axis=1)
y=df['PE']
x

,AT,V,AP,RH
0,8.34,40.77,1010.84,90.01
1,23.64,58.49,1011.40,74.20
2,29.74,56.90,1007.15,41.91
3,19.07,49.69,1007.22,76.79
4,11.80,40.66,1017.13,97.20
...,...,...,...,...
9563,15.12,48.92,1011.80,72.93
9564,33.41,77.95,1010.30,59.72
9565,15.99,43.34,1014.20,78.66
9566,17.65,59.87,1018.58,94.65


In [30]:
y

0       480.48
1       445.75
2       438.76
3       453.09
4       464.43
         ...  
9563    462.59
9564    432.90
9565    465.96
9566    450.93
9567    451.67
Name: PE, Length: 9568, dtype: float64

In [31]:
from sklearn.model_selection import train_test_split
x_tr,x_te,y_tr,y_te=train_test_split(x,y,test_size=0.2,random_state=42)

In [32]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
x_tr_scaled=scaler.fit_transform(x_tr)
x_te_scaled=scaler.transform(x_te)


In [33]:
import torch
import torch.nn as nn
x_tr_tensor=torch.tensor(x_tr_scaled,dtype=torch.float32)
x_te_tensor=torch.tensor(x_te_scaled,dtype=torch.float32)
y_tr_tensor=torch.tensor(y_tr.values,dtype=torch.float32).view(-1,1)
y_te_tensor=torch.tensor(y_te.values,dtype=torch.float32).view(-1,1)

In [34]:
from torch.utils.data import TensorDataset,DataLoader
train_dataset=TensorDataset(x_tr_tensor,y_tr_tensor)
test_dataset=TensorDataset(x_te_tensor,y_te_tensor)

In [35]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=True)

In [36]:
class Ann(nn.Module):
    def __init__(self):
        super(Ann,self).__init__()
        self.model=nn.Sequential(
            nn.Linear(x_tr.shape[1],6),
            nn.ReLU(),
            nn.Linear(6,6),
            nn.ReLU(),
            nn.Linear(6,1)
        )
    def forward(self,x):
        return self.model(x)


In [37]:
import torch.optim as optim
model=Ann()
criteria=nn.MSELoss()
optimizer=optim.Adam(model.parameters())

In [45]:
train_loss=[]
val_loss=[]
n=50
best_val_loss=float("inf")
for epoch in range(n):
    model.train()
    run_loss=0
    for xb,yb in train_loader:
        optimizer.zero_grad()
        op=model(xb)
        loss=criteria(op,yb)
        loss.backward()
        optimizer.step()
        run_loss+=loss.item()
    epoch_train_loss=run_loss/len(train_loader)
    train_loss.append(epoch_train_loss) 

    val_run_loss=0

    model.eval()
    with torch.no_grad():
        for xb,yb in test_loader:
            op=model(xb)
            loss=criteria(op,yb)
            val_run_loss+=loss.item()
    epoch_val_loss=val_run_loss/len(test_loader)
    val_loss.append(epoch_val_loss)   
    print(f"epoch {epoch+1}/{n} ==> train loss = {epoch_train_loss} & val loss = {epoch_val_loss}") 

epoch 1/50 ==> train loss = 20.480875798066457 & val loss = 18.772839132944743
epoch 2/50 ==> train loss = 20.470966897408168 & val loss = 19.442931286493938
epoch 3/50 ==> train loss = 20.566369660695393 & val loss = 18.603757667541505
epoch 4/50 ==> train loss = 20.388092118501664 & val loss = 18.761517588297526
epoch 5/50 ==> train loss = 20.439856402079265 & val loss = 18.65076300303141
epoch 6/50 ==> train loss = 20.50985164642334 & val loss = 18.680370235443114
epoch 7/50 ==> train loss = 20.6064417997996 & val loss = 19.71248861948649
epoch 8/50 ==> train loss = 20.513913011550905 & val loss = 18.693494733174642
epoch 9/50 ==> train loss = 20.467992667357127 & val loss = 18.7259148756663
epoch 10/50 ==> train loss = 20.549414400259653 & val loss = 18.651490465799967
epoch 11/50 ==> train loss = 20.500050648053488 & val loss = 19.2927060286204
epoch 12/50 ==> train loss = 20.570674502849577 & val loss = 18.98652874628703
epoch 13/50 ==> train loss = 20.516335825125378 & val loss 

In [46]:
if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), "best_model.pt")

In [47]:
model.load_state_dict(torch.load("best_model.pt"))

<All keys matched successfully>

In [48]:
model.eval()
with torch.no_grad():
    train_preds = model(x_tr_tensor)
    test_preds = model(x_te_tensor)

    train_mse_loss = criteria(train_preds, y_tr_tensor)
    test_mse_loss = criteria(test_preds, y_te_tensor)

print("Training MSE:", train_mse_loss.item())
print("Testing MSE:", test_mse_loss.item())

Training MSE: 20.4594669342041
Testing MSE: 18.578821182250977


In [49]:
from sklearn.metrics import r2_score

print("r^2 score =", r2_score(y_te, test_preds))

r^2 score = 0.9350718186587326
